### Nota de visualización

La celda siguiente aplica estilos CSS para desactivar el scroll automático de outputs en Jupyter Notebook y JupyterLab, de modo que los resultados se muestren completos sin barra de desplazamiento interna.

In [1]:
from IPython.display import display, HTML

display(HTML("""
<style>
    /* Jupyter Notebook clásico */
    .output_wrapper, .output   { height: auto !important; max-height: none !important; }
    .output_scroll             { height: auto !important; max-height: none !important; }
    /* JupyterLab moderno */
    .jp-OutputArea-child,
    .jp-OutputArea-output      { height: auto !important; max-height: none !important;
                                 overflow: visible !important; }
    .jp-Cell-outputArea        { max-height: none !important; overflow: visible !important; }
    .output_subarea            { max-height: none !important; overflow: visible !important; }
</style>
"""))

# Avance Fase 3 -- Semana 2: Scripts estructurados, recursividad y mediciones de complejidad
**Proyecto:** Pipeline Predictivo y Flujo ETL para Datos Clínicos de Diabetes

| Campo | Detalle |
|---|---|
| **Integrantes** | Gonzalo Bouldres V · Luis Díaz G · Eduardo Contreras |
| **Curso** | MCDI500 -- Herramientas de Software Científico |
| **Programa** | Magíster en Ciencia de Datos e Inteligencia Artificial -- UNAB |
| **Docente** | Docente MCDI500 | Dr. Omar Salinas
| **Fecha** | Junio 2025 |

---


---

## Índice

1. [Importación de librerías y configuración del entorno](#1)
2. [Preprocesamiento -- Reutilización del dataset y las funciones de F2 (II.b)](#2)
   - 2.1 Transformaciones aplicadas y justificación
   - 2.2 Verificación de integridad con asserts
3. [Codificación funcional -- Algoritmos estructurados y recursivos (II.a + II.e)](#3)
   - 3.1 Merge Sort recursivo -- O(n log n)
   - 3.2 Búsqueda binaria recursiva -- O(log n)
   - 3.3 Búsqueda lineal -- O(n)
   - 3.4 Aplicación al dataset de diabetes
4. [Validación técnica -- Caso normal, borde y excepción (II.c)](#4)
5. [Eficiencia y optimización -- Mediciones de complejidad (II.d)](#5)
   - 5.1 Medición 1: Bucle vs Vectorizado -- complejidad temporal O(n)
   - 5.2 Medición 2: Merge Sort vs sorted() -- O(n log n) con distinta constante
   - 5.3 Medición 3: Lista O(n) vs Generador O(1) -- complejidad espacial
   - 5.4 Medición 4: Búsqueda lineal O(n) vs Binaria O(log n)
   - 5.5 Resumen de las cuatro mediciones
6. [Diseño estructurado y escalabilidad -- Auditoría recursiva (II.e)](#6)
7. [Implementación modular -- Clase Preprocesador (III -- POO)](#7)
8. [Documentación de arquitectura del código (IV)](#8)
9. [Bibliografía (VII)](#9)

## 1. Importación de librerías y configuración del entorno <a id="1"></a>

Se importan las librerías del stack científico de Python y los módulos de la biblioteca estándar utilizados para las mediciones de complejidad:

| Librería | Uso en este notebook |
|---|---|
| `pandas`, `numpy` | Manipulación del dataset clínico de diabetes |
| `pathlib.Path` | Gestión de rutas de archivo independiente del sistema operativo |
| `timeit` | Medición de complejidad temporal -- cronometra ejecuciones repetidas |
| `tracemalloc` | Medición de complejidad espacial -- registra el pico de uso de RAM |
| `math` | Cálculo de log2(n) para cuantificar los pasos de la búsqueda binaria |
| `re` | Expresiones regulares en la limpieza de BloodPressure (medición 1) |
| `sklearn.preprocessing` | StandardScaler usado en `escalar_variables()` dentro de `PreprocesadorDiabetes` (Sección 7) |
| `src.utils` | Funciones de F2 reutilizadas directamente y clase POO de F3 |

El módulo `src/utils.py` está un nivel arriba del notebook; se agrega al `sys.path` para que Python lo encuentre sin necesidad de instalarlo.

In [2]:
import sys
import os
import re
import math
import timeit
import tracemalloc

import numpy  as np
import pandas as pd
from   pathlib import Path
from   sklearn.preprocessing import StandardScaler

# Agregar F3/ al path para importar src/utils.py
sys.path.insert(0, os.path.abspath('..'))

# Funciones de F2 y componentes nuevos de F3
from src.utils import (
    cargar_datos,
    explorar_dataset,
    castear_bloodpressure,
    reemplazar_invalidos,
    eliminar_duplicados,
    imputar_mediana,
    crear_variables_derivadas,
    codificar_categoricas,
    escalar_variables,
    validar_dataset_final,
    auditar_carpetas_recursivo,
    PreprocesadorDiabetes,
)

import sklearn
versiones = pd.DataFrame({
    'Componente' : ['Python', 'NumPy', 'Pandas', 'Scikit-Learn'],
    'Versión'    : [
        sys.version.split()[0],
        np.__version__,
        pd.__version__,
        sklearn.__version__,
    ]
})
print("Entorno cargado correctamente")
print(versiones.to_string(index=False))
print()
print("timeit disponible      -- medición de complejidad temporal")
print("tracemalloc disponible -- medición de complejidad espacial")

Entorno cargado correctamente
  Componente Versión
      Python 3.11.15
       NumPy   2.4.6
      Pandas   3.0.3
Scikit-Learn   1.9.0

timeit disponible      -- medición de complejidad temporal
tracemalloc disponible -- medición de complejidad espacial


## 2. Preprocesamiento -- Reutilización del dataset y las funciones de F2 (II.b) <a id="2"></a>

### 2.1 Transformaciones aplicadas y justificación

En esta fase no se reconstruye el pipeline de F2. Se aplican únicamente las transformaciones necesarias para que los algoritmos de ordenamiento y búsqueda operen sobre datos numéricos válidos:

| Función de F2 | Transformación | Por qué se necesita en F3 |
|---|---|---|
| `cargar_datos()` | Lee `diabetes_raw.csv` con separador `;`; lanza `FileNotFoundError` si la ruta no existe | Punto de partida del dataset clínico |
| `castear_bloodpressure()` | Convierte BloodPressure de `str` a `float64` | Sin tipo numérico, `merge_sort` no puede comparar valores |
| `reemplazar_invalidos()` | Ceros imposibles y centinelas → NaN | Ceros en Glucose o BMI distorsionan el orden en cualquier algoritmo de sorting |
| `eliminar_duplicados()` | Elimina filas exactamente iguales | Los duplicados inflan artificialmente n y sesgan las mediciones de complejidad |
| `imputar_mediana()` | NaN -> mediana en Glucose, BloodPressure, BMI, Insulin, Age | Garantiza que las 5 columnas de análisis sean comparables numéricamente. SkinThickness no se incluye: se elimina en la Sección 7. |

El escalamiento Z-score y la codificación OHE no se aplican en esta sección: los algoritmos trabajan sobre los valores originales (años, mg/dL), que son más interpretables en los resultados.

In [3]:
# Rutas
RUTA_RAW = Path('../data/raw/diabetes_raw.csv')
# Paso 1: carga
df_raw = cargar_datos(RUTA_RAW)
print(f"Dataset cargado: {df_raw.shape[0]} filas x {df_raw.shape[1]} columnas")

# Paso 2: transformaciones mínimas para F3
df = df_raw.copy()

df = castear_bloodpressure(df)

df = reemplazar_invalidos(df)

df = eliminar_duplicados(df)

COLS_IMPUTAR = ['Glucose', 'BloodPressure', 'BMI', 'Insulin', 'Age']
df = imputar_mediana(df, COLS_IMPUTAR)

print()
print("=" * 55)
print("DATASET LISTO PARA ANÁLISIS ALGORÍTMICO")
print("=" * 55)
print(f"  Dimensiones     : {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  NaN restantes   : {df[COLS_IMPUTAR].isna().sum().sum()} (en cols de análisis)")
print(f"  Duplicados      : {df.duplicated().sum()}")
print(f"  Tipo BloodPress : {df['BloodPressure'].dtype}")
print()
print("Primeras 5 filas del dataset:")
df.head(5)

# Manejo de errores: verificar que cargar_datos lanza FileNotFoundError si la ruta no existe
try:
    cargar_datos(Path('ruta_inexistente.csv'))
except FileNotFoundError as e:
    print(f"[OK] FileNotFoundError capturado: {e}")
except Exception as e:
    print(f"[OK] Error capturado: {type(e).__name__}: {e}")

Dataset cargado: 2045 filas x 9 columnas
[CASTING BloodPressure]
  Tipo resultante : float64
  NaN en BP       : 20  (de '?' y valores no numéricos)
[CEROS IMPOSIBLES → NaN]
  Glucose             :   13 ceros reemplazados
  BloodPressure       :   70 ceros reemplazados
  BMI                 :   17 ceros reemplazados
  Insulin             :  634 ceros reemplazados

[VALORES CENTINELA / OUTLIERS EXTREMOS → NaN]
  Glucose > 300             :   12 → NaN
  Age > 100 (centinela 150) :   15 → NaN
  BMI > 60                  :    2 → NaN
  BMI < 10 (error digitación):    4 → NaN

  NaN totales tras este paso: 1196
[DUPLICADOS]  Antes: 2045 | Eliminadas: 45 | Después: 2000
[IMPUTACIÓN POR MEDIANA]
  Glucose                     :   25 NaN → mediana = 118.000
  BloodPressure               :   88 NaN → mediana = 72.000
  BMI                         :   23 NaN → mediana = 32.126
  Insulin                     :  834 NaN → mediana = 100.000
  Age                         :   15 NaN → mediana = 29.000


### 2.2 Verificación de integridad con asserts

Antes de trabajar con los algoritmos se comprueba el estado del dataset. Los `assert` sirven como evidencia de ejecución correcta y detienen el notebook con un mensaje claro si alguna condición no se cumple.

> **Nota:** el output de la celda anterior mostrará "NaN restantes en el dataset: 160". Esos NaN corresponden a SkinThickness, que no se imputa en esta sección porque se elimina en el pipeline POO de la Sección 7. Los asserts de la celda siguiente verifican solo las columnas usadas en los algoritmos, donde el resultado es cero NaN.


In [4]:
COLS_ANALISIS = ['Glucose', 'BloodPressure', 'BMI', 'Insulin', 'Age']
assert df[COLS_ANALISIS].isna().sum().sum() == 0, \
    f"Quedan NaN en columnas de análisis: {df[COLS_ANALISIS].isna().sum().to_dict()}"

assert df.duplicated().sum() == 0,     f"Quedan {df.duplicated().sum()} filas duplicadas"

assert df['BloodPressure'].dtype != object,     f"BloodPressure es {df['BloodPressure'].dtype} -- casteo no aplicado"

assert df['Age'].between(0, 100).all(),     "Age contiene valores fuera del rango biológico [0, 100]"

assert (df['Glucose'] > 0).all(),     "Glucose contiene ceros -- reemplazar_invalidos no cubrió todos los casos"

print("[OK] Sin NaN en cols análisis | total:", df[COLS_ANALISIS].isna().sum().sum())
print("[OK] Sin duplicados        | total:", df.duplicated().sum())
print("[OK] BloodPressure float   | tipo :", df['BloodPressure'].dtype)
print("[OK] Age en [0, 100]       | min  :", df['Age'].min(), " max:", df['Age'].max())
print("[OK] Glucose > 0           | min  :", df['Glucose'].min())
print()
print("Dataset listo para los algoritmos de búsqueda y ordenamiento")

[OK] Sin NaN en cols análisis | total: 0
[OK] Sin duplicados        | total: 0
[OK] BloodPressure float   | tipo : float64
[OK] Age en [0, 100]       | min  : 21.0  max: 82.0
[OK] Glucose > 0           | min  : 1.0

Dataset listo para los algoritmos de búsqueda y ordenamiento


## 3. Codificación funcional -- Algoritmos estructurados y recursivos (II.a + II.e) <a id="3"></a>

Se implementan los algoritmos nuevos de la Fase 3. Cada función cumple con los principios de codificación funcional del apartado II.a:

| Principio | Aplicación concreta |
|---|---|
| Una sola responsabilidad | `merge_sort` ordena; `_combinar` fusiona; `busqueda_binaria_recursiva` busca |
| Parámetros tipados | `lista: list`, `objetivo: float`, `inicio: int = 0`, `fin: int = None` |
| Docstring completo | Paradigma, complejidad temporal y espacial, parámetros, retorno, ejemplos |
| Comentarios en puntos críticos | Caso base y caso recursivo marcados explícitamente en el código |
| Retorno explícito | Cada función devuelve un valor concreto verificable de forma independiente |

---

### 3.1 Merge Sort recursivo -- O(n log n)

El Merge Sort divide la lista en dos mitades, ordena cada mitad de forma recursiva y combina los resultados. La profundidad de recursión es log2(n) y cada nivel procesa n elementos, dando O(n log n) en total (Cormen et al., 2022).

In [5]:
def merge_sort(lista: list) -> list:
    """
    Ordena una lista usando Merge Sort recursivo (divide y vencerás).

    Complejidad temporal : O(n log n) -- divide el problema en log2(n) niveles,
                           cada uno con O(n) trabajo de combinación.
    Complejidad espacial : O(n)       -- listas auxiliares en cada combinación.

    Parámetros
    ----------
    lista : list
        Lista de valores numéricos a ordenar. No se modifica; se retorna una copia.

    Retorna
    -------
    list
        Nueva lista con los mismos elementos en orden ascendente.

    Ejemplos
    --------
    >>> merge_sort([3, 1, 4, 1, 5])
    [1, 1, 3, 4, 5]
    >>> merge_sort([])
    []
    """
    # Caso base: lista de 0 ó 1 elemento -- ya está ordenada
    if len(lista) <= 1:
        return lista

    # Caso recursivo: dividir por la mitad y ordenar cada parte
    medio     = len(lista) // 2
    izquierda = merge_sort(lista[:medio])   # mitad izquierda
    derecha   = merge_sort(lista[medio:])   # mitad derecha

    return _combinar(izquierda, derecha)


def _combinar(izq: list, der: list) -> list:
    """
    Fusiona dos listas ordenadas en una sola lista ordenada.

    Compara los primeros elementos de cada lista, toma el menor y avanza.
    Al agotarse una lista, extiende el resultado con el resto de la otra.

    Complejidad temporal : O(n), con n = len(izq) + len(der).
    Complejidad espacial : O(n) por la lista resultado.
    """
    resultado, i, j = [], 0, 0

    while i < len(izq) and j < len(der):
        if izq[i] <= der[j]:
            resultado.append(izq[i]); i += 1
        else:
            resultado.append(der[j]); j += 1

    resultado.extend(izq[i:])
    resultado.extend(der[j:])
    return resultado


print("merge_sort definida    -- O(n log n) temporal, O(n) espacial")
print("_combinar  definida    -- O(n) temporal")
print("Referencia: Cormen et al. (2022), cap. 2")

merge_sort definida    -- O(n log n) temporal, O(n) espacial
_combinar  definida    -- O(n) temporal
Referencia: Cormen et al. (2022), cap. 2


### 3.2 Búsqueda binaria recursiva -- O(log n)

La búsqueda binaria compara el objetivo con el elemento central de la lista. Si no coincide, descarta la mitad donde el objetivo no puede estar y repite el proceso sobre el rango restante. Cada llamada reduce el espacio de búsqueda a la mitad, lo que da O(log n) llamadas en total.

**Requisito previo:** la lista debe estar ordenada. Se aplica `merge_sort()` antes de llamar a esta función.

In [6]:
def busqueda_binaria_recursiva(lista_ordenada: list,
                                objetivo      : float,
                                inicio        : int = 0,
                                fin           : int = None) -> int:
    """
    Busca un valor en una lista ordenada usando recursión.

    En cada llamada descarta la mitad del rango donde el objetivo no puede
    estar, reduciendo el espacio de búsqueda a la mitad en cada paso.

    Complejidad temporal : O(log n) -- número de llamadas = ceil(log2(n)).
    Complejidad espacial : O(log n) -- profundidad máxima de la pila de llamadas.

    Parámetros
    ----------
    lista_ordenada : list
        Lista de valores numéricos en orden ascendente.
    objetivo : float
        Valor a localizar.
    inicio : int
        Límite inferior del rango de búsqueda actual (por defecto 0).
    fin : int
        Límite superior del rango de búsqueda actual (por defecto len - 1).

    Retorna
    -------
    int
        Índice del objetivo en la lista, o -1 si no existe.

    Ejemplos
    --------
    >>> lst = [1, 3, 5, 7, 9]
    >>> busqueda_binaria_recursiva(lst, 5)
    2
    >>> busqueda_binaria_recursiva(lst, 6)
    -1
    """
    if fin is None:
        fin = len(lista_ordenada) - 1

    # Caso base 1: el rango quedó vacío -- el objetivo no está en la lista
    if inicio > fin:
        return -1

    medio = (inicio + fin) // 2

    # Caso base 2: el elemento central coincide con el objetivo
    if lista_ordenada[medio] == objetivo:
        return medio

    # Caso recursivo A: el objetivo está en la mitad derecha
    elif lista_ordenada[medio] < objetivo:
        return busqueda_binaria_recursiva(lista_ordenada, objetivo, medio + 1, fin)

    # Caso recursivo B: el objetivo está en la mitad izquierda
    else:
        return busqueda_binaria_recursiva(lista_ordenada, objetivo, inicio, medio - 1)


print("busqueda_binaria_recursiva definida -- O(log n) temporal, O(log n) espacial")

busqueda_binaria_recursiva definida -- O(log n) temporal, O(log n) espacial


### 3.3 Búsqueda lineal -- O(n)

Implementación iterativa que recorre la lista posición a posición. No exige que la lista esté ordenada. Se usa como referencia para comparar empíricamente con la búsqueda binaria en la Medición 4.

In [7]:
def busqueda_lineal(lista: list, objetivo: float) -> int:
    """
    Localiza el primer elemento igual al objetivo recorriendo la lista en orden.

    No requiere lista ordenada. En el peor caso examina todos los n elementos.

    Complejidad temporal : O(n).
    Complejidad espacial : O(1) -- no requiere estructuras auxiliares.

    Parámetros
    ----------
    lista    : list   Lista de valores a recorrer.
    objetivo : float  Valor a buscar.

    Retorna
    -------
    int
        Índice del primer elemento igual al objetivo, o -1 si no existe.
    """
    for i, valor in enumerate(lista):
        if valor == objetivo:
            return i
    return -1


print("busqueda_lineal definida -- O(n) temporal, O(1) espacial")

busqueda_lineal definida -- O(n) temporal, O(1) espacial


### 3.4 Aplicación al dataset de diabetes

Se extraen `Age` y `Glucose` como listas Python y se aplican los algoritmos sobre los valores originales (años, mg/dL), antes del escalamiento, para que los resultados sean directamente interpretables.

In [8]:
edades   = df['Age'].dropna().tolist()     # valores en años
glucosas = df['Glucose'].dropna().tolist() # valores en mg/dL

print(f"n edades   : {len(edades)}  | rango crudo: {min(edades):.0f} - {max(edades):.0f} años")
print(f"n glucosas : {len(glucosas)} | rango crudo: {min(glucosas):.0f} - {max(glucosas):.0f} mg/dL")
print()

# Merge Sort sobre Age
edades_ordenadas = merge_sort(edades)

print("=" * 58)
print("MERGE SORT RECURSIVO -- Columna 'Age' (años)")
print("=" * 58)
print(f"  n (registros ordenados)  : {len(edades_ordenadas)}")
print(f"  Valor mínimo             : {edades_ordenadas[0]:.1f} años")
print(f"  Valor máximo             : {edades_ordenadas[-1]:.1f} años")
print(f"  Percentil 25 (índice)    : {edades_ordenadas[len(edades_ordenadas)//4]:.1f} años")
print(f"  Mediana (índice central) : {edades_ordenadas[len(edades_ordenadas)//2]:.1f} años")
print(f"  Las 5 edades menores     : {[round(v,1) for v in edades_ordenadas[:5]]}")
print(f"  Las 5 edades mayores     : {[round(v,1) for v in edades_ordenadas[-5:]]}")
print()

# Búsqueda binaria sobre Glucose
glucosas_ordenadas = merge_sort(glucosas)

# Buscar la mediana del dataset
idx_mediana    = len(glucosas_ordenadas) // 2
objetivo_gluc  = glucosas_ordenadas[idx_mediana]
idx_encontrado = busqueda_binaria_recursiva(glucosas_ordenadas, objetivo_gluc)

print("=" * 58)
print("BÚSQUEDA BINARIA RECURSIVA -- Columna 'Glucose' (mg/dL)")
print("=" * 58)
print(f"  n (registros ordenados)  : {len(glucosas_ordenadas)}")
print(f"  Valor buscado            : {objetivo_gluc:.1f} mg/dL (mediana del dataset)")
print(f"  Índice hallado           : {idx_encontrado}")
print(f"  Verificación             : glucosas_ordenadas[{idx_encontrado}] = {glucosas_ordenadas[idx_encontrado]:.1f} mg/dL")
print(f"  Valor coincide           : {abs(glucosas_ordenadas[idx_encontrado] - objetivo_gluc) < 1e-9}")
print(f"  Pasos realizados         : <= {math.ceil(math.log2(len(glucosas_ordenadas)))} comparaciones (log2({len(glucosas_ordenadas)}))")
print()
print("Algoritmos aplicados correctamente sobre el dataset de diabetes")

n edades   : 2000  | rango crudo: 21 - 82 años
n glucosas : 2000 | rango crudo: 1 - 199 mg/dL

MERGE SORT RECURSIVO -- Columna 'Age' (años)
  n (registros ordenados)  : 2000
  Valor mínimo             : 21.0 años
  Valor máximo             : 82.0 años
  Percentil 25 (índice)    : 24.0 años
  Mediana (índice central) : 29.0 años
  Las 5 edades menores     : [21.0, 21.0, 21.0, 21.0, 21.0]
  Las 5 edades mayores     : [72.0, 80.0, 81.0, 81.0, 82.0]

BÚSQUEDA BINARIA RECURSIVA -- Columna 'Glucose' (mg/dL)
  n (registros ordenados)  : 2000
  Valor buscado            : 118.0 mg/dL (mediana del dataset)
  Índice hallado           : 999
  Verificación             : glucosas_ordenadas[999] = 118.0 mg/dL
  Valor coincide           : True
  Pasos realizados         : <= 11 comparaciones (log2(2000))

Algoritmos aplicados correctamente sobre el dataset de diabetes


## 4. Validación técnica -- Caso normal, borde y excepción (II.c) <a id="4"></a>

Se verifican las tres funciones nuevas en tres escenarios distintos:

| Escenario | Qué se prueba |
|---|---|
| **Caso normal** | Entrada típica con varios elementos y resultado esperado conocido |
| **Caso borde** | Lista vacía, un elemento, ya ordenada, orden inverso, repetidos |
| **Caso excepción** | Elemento inexistente (retorno -1) y tipos incompatibles (TypeError) |

Cada prueba usa `assert` para detener el notebook si el resultado no es el esperado, y `try/except` para capturar excepciones previstas.

In [9]:
print("=" * 55)
print("Caso normal")
print("=" * 55)

lista_prueba   = [5, 2, 8, 1, 9, 3, 7, 4, 6, 0]
resultado_sort = merge_sort(lista_prueba)
assert resultado_sort == sorted(lista_prueba), f"merge_sort: resultado incorrecto: {resultado_sort}"
print(f"merge_sort({lista_prueba})")
print(f"        -> {resultado_sort}  OK")

lista_bin = merge_sort([3, 1, 4, 1, 5, 9, 2, 6, 5, 3])
idx_bin   = busqueda_binaria_recursiva(lista_bin, 6)
assert idx_bin >= 0 and lista_bin[idx_bin] == 6
print(f"\nbusqueda_binaria_recursiva(lista_ordenada, 6)")
print(f"        -> índice {idx_bin}, valor {lista_bin[idx_bin]}  OK")

idx_lin = busqueda_lineal([10, 30, 20, 50, 40], 50)
assert idx_lin == 3
print(f"\nbusqueda_lineal([10, 30, 20, 50, 40], 50)")
print(f"        -> índice {idx_lin}, valor 50  OK")

print()
print("=" * 55)
print("Casos borde")
print("=" * 55)

assert merge_sort([]) == []
print(f"merge_sort([])        -> []   OK  (lista vacía)")

assert merge_sort([42]) == [42]
print(f"merge_sort([42])      -> [42] OK  (un elemento)")

ya_ord = [1, 2, 3, 4, 5]
assert merge_sort(ya_ord) == ya_ord
print(f"merge_sort({ya_ord}) -> {merge_sort(ya_ord)} OK  (ya ordenada)")

inv = [9, 7, 5, 3, 1]
assert merge_sort(inv) == sorted(inv)
print(f"merge_sort({inv})  -> {merge_sort(inv)} OK  (orden inverso)")

rep = [3, 1, 3, 2, 1]
assert merge_sort(rep) == sorted(rep)
print(f"merge_sort({rep})     -> {merge_sort(rep)} OK  (repetidos)")

lista_ext = merge_sort([10, 20, 30, 40, 50])
assert lista_ext[busqueda_binaria_recursiva(lista_ext, 10)] == 10
print(f"binaria(lista, 10)   -> índice {busqueda_binaria_recursiva(lista_ext, 10)} OK  (primer elemento)")

assert lista_ext[busqueda_binaria_recursiva(lista_ext, 50)] == 50
print(f"binaria(lista, 50)   -> índice {busqueda_binaria_recursiva(lista_ext, 50)} OK  (último elemento)")

print()
print("=" * 55)
print("Caso excepción / no encontrado")
print("=" * 55)

idx_no = busqueda_binaria_recursiva(merge_sort([10, 20, 30, 40, 50]), 99)
assert idx_no == -1
print(f"binaria([10..50], 99)      -> {idx_no}  OK  (no encontrado)")

idx_v = busqueda_binaria_recursiva([], 5)
assert idx_v == -1
print(f"binaria([], 5)             -> {idx_v}  OK  (lista vacía)")

idx_ln = busqueda_lineal([10, 20, 30], 99)
assert idx_ln == -1
print(f"lineal([10,20,30], 99)     -> {idx_ln}  OK  (no encontrado)")

try:
    merge_sort([1, 'dos', 3])
except TypeError as e:
    print(f"merge_sort([1,'dos',3])    -> TypeError: {e}  OK")

print()
print("Todas las validaciones pasaron correctamente")

Caso normal
merge_sort([5, 2, 8, 1, 9, 3, 7, 4, 6, 0])
        -> [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  OK

busqueda_binaria_recursiva(lista_ordenada, 6)
        -> índice 8, valor 6  OK

busqueda_lineal([10, 30, 20, 50, 40], 50)
        -> índice 3, valor 50  OK

Casos borde
merge_sort([])        -> []   OK  (lista vacía)
merge_sort([42])      -> [42] OK  (un elemento)
merge_sort([1, 2, 3, 4, 5]) -> [1, 2, 3, 4, 5] OK  (ya ordenada)
merge_sort([9, 7, 5, 3, 1])  -> [1, 3, 5, 7, 9] OK  (orden inverso)
merge_sort([3, 1, 3, 2, 1])     -> [1, 1, 2, 3, 3] OK  (repetidos)
binaria(lista, 10)   -> índice 0 OK  (primer elemento)
binaria(lista, 50)   -> índice 4 OK  (último elemento)

Caso excepción / no encontrado
binaria([10..50], 99)      -> -1  OK  (no encontrado)
binaria([], 5)             -> -1  OK  (lista vacía)
lineal([10,20,30], 99)     -> -1  OK  (no encontrado)
merge_sort([1,'dos',3])    -> TypeError: '<=' not supported between instances of 'str' and 'int'  OK

Todas las validaciones pasaro

## 5. Eficiencia y optimización -- Mediciones de complejidad (II.d) <a id="5"></a>

La notación **Big-O** expresa el crecimiento del tiempo de ejecución o del uso de memoria en función del tamaño de la entrada n, independientemente del hardware. Permite comparar algoritmos de forma objetiva (Cormen et al., 2022, cap. 2-4).

| Notación | Nombre | Descripción | Ejemplo en este notebook |
|---|---|---|---|
| O(1) | Constante | No crece con n | Acceso a un elemento por índice |
| O(log n) | Logarítmica | Descarta la mitad del rango en cada paso | Búsqueda binaria |
| O(n) | Lineal | Recorre todos los n elementos | Búsqueda lineal, vectorizado Pandas |
| O(n log n) | Linealítmica | Divide el problema en log2(n) niveles | Merge Sort |
| O(n^2) | Cuadrática | Doble bucle anidado | Bubble Sort |

**Herramientas usadas:**

| Herramienta | Mide | Aplicación |
|---|---|---|
| `timeit.timeit()` | Complejidad temporal (tiempo de CPU) | Mediciones 1, 2 y 4 |
| `tracemalloc` | Complejidad espacial (pico de RAM) | Medición 3 |

Cada medición tiene su propia celda de código seguida de una celda de interpretación con la lectura del resultado en términos de Big-O.

---

### 5.1 Medición 1 -- Bucle Python vs Operación Vectorizada

`BloodPressure` llega como `str` en el CSV crudo. Comparamos dos formas de limpiarla: un bucle Python que procesa cada fila individualmente y una operación `str.replace` vectorizada de Pandas.

Las dos tienen la misma complejidad O(n), pero la vectorizada delega el procesamiento a rutinas en C, lo que reduce la constante multiplicativa k.

In [10]:
df_crudo = pd.read_csv('../data/raw/diabetes_raw.csv', sep=';')

# Implementación A: bucle Python
def limpiar_con_bucle(df: pd.DataFrame) -> list:
    """
    Limpia BloodPressure recorriendo cada fila con un bucle explícito.
    Complejidad temporal: O(n) -- n iteraciones del intérprete Python.
    Complejidad espacial: O(n) -- lista resultado de n elementos.
    """
    marcas = []
    for idx in range(len(df)):
        val    = str(df.loc[idx, 'BloodPressure'])
        limpio = re.sub(r'[^\d]', '', val)    # eliminar todo lo que no sea dígito
        marcas.append(limpio)
    return marcas

# Implementación B: vectorizado Pandas
def limpiar_vectorizado(df: pd.DataFrame) -> pd.Series:
    """
    Limpia BloodPressure con str.replace vectorizado de Pandas.
    Complejidad temporal: O(n) -- mismo orden que el bucle, pero en C.
    Complejidad espacial: O(n) -- Series resultado de n elementos.
    """
    return df['BloodPressure'].astype(str).str.replace(r'[^\d]', '', regex=True)

N_REPS = 10
t_bucle       = timeit.timeit(lambda: limpiar_con_bucle(df_crudo),   number=N_REPS)
t_vectorizado = timeit.timeit(lambda: limpiar_vectorizado(df_crudo), number=N_REPS)
factor_1      = t_bucle / t_vectorizado

print("=" * 62)
print("MEDICIÓN 1 -- Limpieza de BloodPressure (n = {})".format(len(df_crudo)))
print("  Bucle Python (O(n) · intérprete) vs Vectorizado Pandas (O(n) · C)")
print("=" * 62)
print(f"  Repeticiones             : {N_REPS}")
print(f"  Tiempo bucle             : {t_bucle:.4f} s  (total {N_REPS} reps)")
print(f"  Tiempo vectorizado       : {t_vectorizado:.4f} s  (total {N_REPS} reps)")
print(f"  Tiempo bucle  / rep      : {t_bucle/N_REPS*1000:.2f} ms")
print(f"  Tiempo vector / rep      : {t_vectorizado/N_REPS*1000:.2f} ms")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Factor de aceleración    : x{factor_1:.1f}  (vectorizado es {factor_1:.0f}x más rápido)")
print("=" * 62)

MEDICIÓN 1 -- Limpieza de BloodPressure (n = 2045)
  Bucle Python (O(n) · intérprete) vs Vectorizado Pandas (O(n) · C)
  Repeticiones             : 10
  Tiempo bucle             : 1.7640 s  (total 10 reps)
  Tiempo vectorizado       : 0.0332 s  (total 10 reps)
  Tiempo bucle  / rep      : 176.40 ms
  Tiempo vector / rep      : 3.32 ms
  ─────────────────────────────────────────────────────────
  Factor de aceleración    : x53.1  (vectorizado es 53x más rápido)


**Interpretación -- Medición 1**

Ambas implementaciones son **O(n)**: el tiempo crece linealmente con el número de filas. La diferencia está en la constante k:

- Bucle Python: tiempo ~ k_Python x n, k_Python ~ 0.030 ms/fila
- Vectorizado:  tiempo ~ k_C x n,      k_C ~ 0.001 ms/fila

Para n = 1.000.000 de filas, el bucle tomaría ~30 s y el vectorizado ~1 s. La clase de complejidad no cambia, pero la constante es crítica en producción (McKinney, 2022, cap. 6).

---

### 5.2 Medición 2 -- Merge Sort recursivo vs sorted()

Comparamos `merge_sort` (Python puro) con `sorted()` de Python, que usa **Timsort** implementado en C. Los dos son O(n log n).

In [11]:
edades_lista = df['Age'].dropna().tolist()     # lista de trabajo (n ~2.000)
n_age        = len(edades_lista)

N_REPS2  = 50
t_merge  = timeit.timeit(lambda: merge_sort(edades_lista.copy()), number=N_REPS2)
t_sorted = timeit.timeit(lambda: sorted(edades_lista),             number=N_REPS2)
factor_2 = t_merge / t_sorted

assert merge_sort(edades_lista) == sorted(edades_lista),     "merge_sort y sorted() producen resultados distintos"

print("=" * 62)
print(f"MEDICIÓN 2 -- Ordenamiento de 'Age' (n = {n_age})")
print("  merge_sort() Python puro vs sorted() Python (Timsort en C)")
print("=" * 62)
print(f"  Repeticiones              : {N_REPS2}")
print(f"  Tiempo merge_sort         : {t_merge:.4f} s  (total {N_REPS2} reps)")
print(f"  Tiempo sorted()           : {t_sorted:.4f} s  (total {N_REPS2} reps)")
print(f"  Tiempo merge_sort / rep   : {t_merge/N_REPS2*1000:.2f} ms")
print(f"  Tiempo sorted()   / rep   : {t_sorted/N_REPS2*1000:.2f} ms")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Factor de diferencia      : x{factor_2:.1f}  (sorted() es {factor_2:.0f}x más rápido)")
print(f"  ¿Mismo resultado?         : {merge_sort(edades_lista) == sorted(edades_lista)}")
print("=" * 62)

MEDICIÓN 2 -- Ordenamiento de 'Age' (n = 2000)
  merge_sort() Python puro vs sorted() Python (Timsort en C)
  Repeticiones              : 50
  Tiempo merge_sort         : 0.8207 s  (total 50 reps)
  Tiempo sorted()           : 0.0162 s  (total 50 reps)
  Tiempo merge_sort / rep   : 16.41 ms
  Tiempo sorted()   / rep   : 0.32 ms
  ─────────────────────────────────────────────────────────
  Factor de diferencia      : x50.8  (sorted() es 51x más rápido)
  ¿Mismo resultado?         : True


**Interpretación -- Medición 2**

Los dos algoritmos son **O(n log n)** -- la curva de crecimiento es idéntica. La diferencia en tiempo absoluto se debe a la constante k:

- `merge_sort()` en Python: k ~ 0.03 ms por operación elemental
- `sorted()` Timsort en C:  k ~ 0.001 ms por operación elemental

Timsort también aprovecha secuencias ya ordenadas internamente, lo que lo hace cercano a O(n) en datos casi ordenados. Para n = 1.000.000, el factor de diferencia se mantiene (~x10-20) porque la complejidad es la misma. Usamos `merge_sort` en este notebook para demostrar el paradigma recursivo, no por eficiencia absoluta.

---

### 5.3 Medición 3 -- Lista O(n) vs Generador O(1) -- complejidad espacial

Medimos el pico de uso de RAM con `tracemalloc`. Comparamos una lista por comprensión, que crea todos los valores en RAM de una vez, contra una expresión generadora, que los computa uno a uno.

In [12]:
N_ELEMENTOS = 10_000   # n para la comparación de estructuras

# Lista por comprensión -- O(n) espacial
tracemalloc.start()
valores_lista = [x**2 for x in range(N_ELEMENTOS)]   # guarda n valores en RAM
_, pico_lista = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Expresión generadora -- O(1) espacial
tracemalloc.start()
suma_gen = sum(x**2 for x in range(N_ELEMENTOS))      # computa uno a uno
_, pico_gen = tracemalloc.get_traced_memory()
tracemalloc.stop()

# merge_sort vs sorted() -- pico de RAM
tracemalloc.start()
_ = merge_sort(df['Age'].dropna().tolist())
_, pico_merge_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
_ = sorted(df['Age'].dropna().tolist())
_, pico_sorted_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("=" * 62)
print(f"MEDICIÓN 3a -- Lista vs Generador  (n = {N_ELEMENTOS:,})")
print("=" * 62)
print(f"  Lista por comprensión  [O(n) espacial] : {pico_lista/1024:.1f} KB")
print(f"  Expresión generadora   [O(1) espacial] : {pico_gen/1024:.1f} KB")
factor_mem = pico_lista / max(pico_gen, 1)
print(f"  Reducción de memoria                   : x{factor_mem:.0f}")
print(f"  Ambas calculan la misma suma            : {sum(valores_lista) == suma_gen}")
print()
print("=" * 62)
print(f"MEDICIÓN 3b -- merge_sort vs sorted() sobre 'Age'  (n = {len(df)})")
print("=" * 62)
print(f"  merge_sort() -- recursivo  [O(n) RAM] : {pico_merge_mem/1024:.1f} KB")
print(f"  sorted()     -- Timsort    [O(n) RAM] : {pico_sorted_mem/1024:.1f} KB")
print(f"  Overhead recursión merge_sort        : +{(pico_merge_mem-pico_sorted_mem)/1024:.1f} KB  (pilas de llamadas)")
print("=" * 62)

MEDICIÓN 3a -- Lista vs Generador  (n = 10,000)
  Lista por comprensión  [O(n) espacial] : 408.0 KB
  Expresión generadora   [O(1) espacial] : 16.2 KB
  Reducción de memoria                   : x25
  Ambas calculan la misma suma            : True

MEDICIÓN 3b -- merge_sort vs sorted() sobre 'Age'  (n = 2000)
  merge_sort() -- recursivo  [O(n) RAM] : 242.2 KB
  sorted()     -- Timsort    [O(n) RAM] : 84.3 KB
  Overhead recursión merge_sort        : +157.9 KB  (pilas de llamadas)


**Interpretación -- Medición 3**

**3a -- Lista vs Generador:**
La lista O(n) reserva memoria para n = 10.000 enteros a la vez. El generador O(1) solo mantiene el estado del iterador (unos pocos bytes) y computa cada valor cuando `sum()` lo solicita. Para n = 1.000.000, la lista ocuparía ~8 MB; el generador usaría siempre los mismos ~200 bytes.

**3b -- merge_sort vs sorted() en RAM:**
Los dos son O(n) en memoria porque necesitan listas auxiliares durante la combinación. `merge_sort` acumula algo más de RAM porque cada llamada recursiva mantiene en la pila las sublistas de izquierda y derecha; `sorted()` usa una implementación en C más compacta.

La complejidad espacial determina si un algoritmo puede procesar datasets de decenas de millones de registros sin agotar la memoria disponible.

---

### 5.4 Medición 4 -- Búsqueda lineal O(n) vs Búsqueda binaria O(log n)

Esta es la comparación más significativa de la fase porque involucra **complejidades distintas**: la brecha no es una constante k, sino que crece con n. Medimos el peor caso: buscar el último elemento de la lista.

In [13]:
glucosas_ord = merge_sort(df['Glucose'].dropna().tolist())
n_g          = len(glucosas_ord)

# Peor caso: último elemento
objetivo_peor = glucosas_ord[-1]

N_REPS4  = 1000
t_lineal  = timeit.timeit(
    lambda: busqueda_lineal(glucosas_ord, objetivo_peor), number=N_REPS4)
t_binaria = timeit.timeit(
    lambda: busqueda_binaria_recursiva(glucosas_ord, objetivo_peor), number=N_REPS4)
factor_4 = t_lineal / t_binaria

pasos_lineal  = n_g                              # O(n) -- recorre todo en peor caso
pasos_binaria = math.ceil(math.log2(n_g))        # O(log n)

idx_l = busqueda_lineal(glucosas_ord, objetivo_peor)
idx_b = busqueda_binaria_recursiva(glucosas_ord, objetivo_peor)
assert glucosas_ord[idx_l] == glucosas_ord[idx_b] == objetivo_peor,     "Las dos búsquedas retornaron resultados distintos"

print("=" * 62)
print(f"MEDICIÓN 4 -- Búsqueda sobre Glucose ordenado  (n = {n_g})")
print("  Búsqueda lineal O(n) vs Búsqueda binaria O(log n)")
print("  Escenario: peor caso -- búsqueda del último elemento")
print("=" * 62)
print(f"  Tiempo búsqueda lineal  ({N_REPS4} reps)  : {t_lineal:.4f} s")
print(f"  Tiempo búsqueda binaria ({N_REPS4} reps)  : {t_binaria:.4f} s")
print(f"  Tiempo lineal  / rep                    : {t_lineal/N_REPS4*1e6:.2f} us")
print(f"  Tiempo binaria / rep                    : {t_binaria/N_REPS4*1e6:.2f} us")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Factor de aceleración                   : x{factor_4:.1f}")
print()
print(f"  Comparaciones en peor caso:")
print(f"    Lineal   O(n)     : {pasos_lineal} comparaciones")
print(f"    Binaria  O(log n) : {pasos_binaria} comparaciones  [log2({n_g}) = {pasos_binaria}]")
print()
print(f"  Proyección para n = 1.000.000 registros:")
print(f"    Lineal   O(n)     : 1.000.000 comparaciones")
print(f"    Binaria  O(log n) : {math.ceil(math.log2(1_000_000))} comparaciones")
print("=" * 62)

MEDICIÓN 4 -- Búsqueda sobre Glucose ordenado  (n = 2000)
  Búsqueda lineal O(n) vs Búsqueda binaria O(log n)
  Escenario: peor caso -- búsqueda del último elemento
  Tiempo búsqueda lineal  (1000 reps)  : 0.3602 s
  Tiempo búsqueda binaria (1000 reps)  : 0.0062 s
  Tiempo lineal  / rep                    : 360.24 us
  Tiempo binaria / rep                    : 6.23 us
  ─────────────────────────────────────────────────────────
  Factor de aceleración                   : x57.8

  Comparaciones en peor caso:
    Lineal   O(n)     : 2000 comparaciones
    Binaria  O(log n) : 11 comparaciones  [log2(2000) = 11]

  Proyección para n = 1.000.000 registros:
    Lineal   O(n)     : 1.000.000 comparaciones
    Binaria  O(log n) : 20 comparaciones


**Interpretación -- Medición 4**

La diferencia es **estructural**: no depende del lenguaje ni del hardware, sino del número de comparaciones que cada algoritmo realiza por definición.

Para n = 2.000:
- Búsqueda lineal: hasta **2.000 comparaciones** en el peor caso
- Búsqueda binaria: hasta **~11 comparaciones** -- log2(2.000) ~ 11

El factor de aceleración crece con n:

| n | Lineal | Binaria | Factor |
|---|---|---|---|
| 1.000 | 1.000 | 10 | x100 |
| 10.000 | 10.000 | 14 | x714 |
| 1.000.000 | 1.000.000 | 20 | x50.000 |

Para datasets de producción con millones de registros, la búsqueda binaria es la única opción viable cuando la lista está ordenada.

---

### 5.5 Resumen de las cuatro mediciones

| Experimento | Implementación A | Implementación B | Complejidad A | Complejidad B | Factor observado |
|---|---|---|---|---|---|
| Limpieza `BloodPressure` | Bucle Python | Vectorizado Pandas | O(n) | O(n) | x30-100 (constante k) |
| Ordenamiento `Age` | `merge_sort` Python | `sorted()` Timsort | O(n log n) | O(n log n) | x10-20 (constante k) |
| Suma de cuadrados | Lista O(n) | Generador O(1) | O(n) RAM | O(1) RAM | x100-400 (uso de RAM) |
| Búsqueda `Glucose` | Lineal O(n) | Binaria O(log n) | O(n) | O(log n) | x5-50 (crece con n) |

Cuando la complejidad es igual, la diferencia la produce la constante k: las implementaciones en C son ~30-100x más rápidas que Python puro. Cuando la complejidad es distinta, la brecha aumenta con n y es independiente del lenguaje. La complejidad espacial define si el algoritmo escala a datasets de millones de filas sin agotar la RAM.

## 6. Diseño estructurado y escalabilidad -- Auditoría recursiva (II.e) <a id="6"></a>

### 6.1 Función recursiva de infraestructura

`auditar_carpetas_recursivo()` recorre el árbol de directorios del proyecto y lista los archivos relevantes (.csv, .ipynb, .py) en cada nivel. El patrón recursivo es directo:

- **Caso base:** directorio sin subdirectorios -- imprime sus archivos y retorna.
- **Caso recursivo:** hay subdirectorios -- se llama a sí misma con `profundidad + 1`, incrementando la indentación en cada nivel.

La función está implementada en `F3/src/utils.py`.

In [14]:
print("Estructura de directorios del proyecto (auditoría recursiva):")
print("─" * 55)
auditar_carpetas_recursivo('../../')
print("─" * 55)
print("Auditoría completada")

Estructura de directorios del proyecto (auditoría recursiva):
───────────────────────────────────────────────────────
[F1]
  [data]
    [processed]
    [raw]
      [.ipynb_checkpoints]
          diabetes_raw-checkpoint.csv
        diabetes_raw.csv
  [docs]
  [notebooks]
      F1_Definicion.ipynb
  [reports]
  [src]
[F2]
  [data]
    [processed]
    [raw]
        diabetes_raw.csv
  [docs]
  [notebooks]
      F2_Ejecucion.ipynb
      F2_Preprocesamiento.ipynb
  [reports]
  [src]
      utils.py
[F3]
  [data]
    [processed]
        diabetes_procesado.csv
    [raw]
        diabetes_raw.csv
  [docs]
  [notebooks]
    [.ipynb_checkpoints]
        F3_Preprocesamiento-checkpoint.ipynb
        F3_PreprocesamientoV2-checkpoint.ipynb
      F3_Preprocesamiento.ipynb
      F3_PreprocesamientoV2.ipynb
  [src]
      utils.py
[F4]
───────────────────────────────────────────────────────
Auditoría completada


### 6.2 Organización modular en tres capas

El código de la Fase 3 está distribuido en tres capas con responsabilidades separadas:

```
┌─────────────────────────────────────────────────────────────┐
│ Capa 1 -- Funciones de F2 (reutilizadas sin cambios)         │
│   cargar_datos · castear_bloodpressure · reemplazar_invalidos│
│   eliminar_duplicados · imputar_mediana · exportar_dataset  │
│   Ubicación: F3/src/utils.py                                │
├─────────────────────────────────────────────────────────────┤
│ Capa 2 -- Algoritmos nuevos de F3                            │
│   merge_sort() · _combinar()                                │
│   busqueda_binaria_recursiva() · busqueda_lineal()          │
│   Ubicación: F3/notebooks/F3_Preprocesamiento.ipynb         │
├─────────────────────────────────────────────────────────────┤
│ Capa 3 -- POO (complementaria)                               │
│   PreprocesadorDiabetes -- pipeline encapsulado              │
│   auditar_carpetas_recursivo -- recursión de infraestructura │
│   Ubicación: F3/src/utils.py                                │
└─────────────────────────────────────────────────────────────┘
```

### 6.3 Escalabilidad hacia F4

La arquitectura de tres capas permite incorporar el modelado predictivo de F4 sin modificar lo ya implementado:

- Capa 1: agregar `preparar_train_test(df, target)` para dividir el dataset manteniendo la proporción de clases.
- Capa 2: `merge_sort` puede ordenar probabilidades predichas para construir curvas ROC; `busqueda_binaria_recursiva` puede localizar el umbral de clasificación que maximiza el F1-score.
- Capa 3: crear `PreprocesadorML(PreprocesadorDiabetes)` que herede el pipeline y agregue métodos para ajustar modelos y evaluar métricas.

## 7. Implementación modular -- Clase Preprocesador (III -- POO) <a id="7"></a>

### 7.1 Encapsulamiento -- clase base

`PreprocesadorDiabetes` mantiene el DataFrame en `self.df` y expone métodos que retornan `self` (implementado en `F3/src/utils.py`) para permitir method chaining. Esto elimina variables globales del notebook y centraliza el estado del pipeline.

La clase elimina `SkinThickness` porque el 32.6% de sus registros son ceros biológicamente imposibles -- imputar sobre esa proporción introduciría sesgo significativo.

---

### 7.2 Herencia y polimorfismo -- Transformadores intercambiables

Cuando un paso del pipeline tiene variantes (imputar por mediana o por media; escalar con Z-score o Min-Max), se define un contrato común en la clase base `Transformador` y cada variante lo implementa en su subclase. Un mismo bucle ejecuta cualquier combinación sin saber cuál es cuál -- eso es **polimorfismo**.

| Clase | Hereda de | Comportamiento en `aplicar(df)` |
|---|---|---|
| `Transformador` | -- | Contrato abstracto (NotImplementedError) |
| `ImputarMediana` | `Transformador` | Rellena NaN con la mediana de cada columna |
| `EscalarZScore` | `Transformador` | Estandariza a media=0, std=1 (Z-score) |


> **Nota sobre los tres procesamientos del notebook:** la Seccion 2 aplica el pipeline funcional minimo para los algoritmos de busqueda y ordenamiento. La Seccion 7 demuestra dos enfoques de POO sobre el mismo dataset crudo: primero el patron polimorfismo con `ImputarMediana` y `EscalarZScore`, y luego el pipeline completo de produccion con `PreprocesadorDiabetes` que exporta el dataset final a `F3/data/processed/`.


---

### 7.3 Composicion -- clase Pipeline que une F2 con F3

La guia propone unir las transformaciones de F2 con el nucleo algoritmico de F3 en una clase `Pipeline` que recibe una lista de etapas y las ejecuta en orden. Cada etapa implementa `aplicar(df)` -- el pipeline no sabe cual es cual, solo las ejecuta. Eso es **composicion** (el pipeline tiene etapas) mas **polimorfismo** (mismo metodo, distinto comportamiento).

| Componente | Rol |
|---|---|
| `Pipeline` | Orquestador -- recibe etapas y las ejecuta en orden |
| `ImputarMediana` | Etapa de preprocesamiento F2 |
| `EscalarZScore` | Etapa de preprocesamiento F2 |
| `OrdenarColumna` | Etapa del nucleo algoritmico F3 -- aplica merge_sort sobre una columna |


In [15]:
from abc import ABC, abstractmethod

# Clase base -- contrato comun
class Transformador(ABC):
    """
    Contrato comun para todas las etapas del pipeline.
    Cada subclase implementa aplicar() con su propio comportamiento.
    """
    @abstractmethod
    def aplicar(self, df):
        """Aplica la transformacion al DataFrame y lo retorna."""
        ...


# Subclase 1 -- herencia: imputa NaN con la mediana
class ImputarMediana(Transformador):
    """
    Rellena NaN con la mediana de cada columna indicada.
    Se elige mediana sobre media porque las distribuciones clinicas
    tienen outliers extremos que distorsionarian el promedio.
    Hereda de: Transformador
    """
    def __init__(self, columnas: list):
        self.columnas = columnas

    def aplicar(self, df):
        for col in self.columnas:
            if col in df.columns:
                df[col] = df[col].fillna(df[col].median())
        return df


# Subclase 2 -- herencia: estandariza con Z-score
class EscalarZScore(Transformador):
    """
    Estandariza columnas a media=0 y std=1.
    Se elige Z-score sobre Min-Max porque outliers residuales
    en Min-Max comprimen el rango util de los datos.
    Hereda de: Transformador
    """
    def __init__(self, columnas: list):
        self.columnas = columnas

    def aplicar(self, df):
        for col in self.columnas:
            if col in df.columns:
                media = df[col].mean()
                std   = df[col].std()
                df[col] = (df[col] - media) / std
        return df


# Polimorfismo: mismo bucle, distinto comportamiento
COLS_IMPUTAR = ['Glucose', 'BloodPressure', 'BMI', 'Insulin', 'Age']
COLS_ESCALAR = ['Pregnancies', 'Glucose', 'BloodPressure', 'Insulin',
                'BMI', 'DiabetesPedigreeFunction', 'Age']

transformadores = [
    ImputarMediana(COLS_IMPUTAR),
    EscalarZScore(COLS_ESCALAR),
]

df_poly = df_raw.copy()
df_poly = castear_bloodpressure(df_poly)
df_poly = reemplazar_invalidos(df_poly)
df_poly = eliminar_duplicados(df_poly)
df_poly = df_poly.drop(columns=['SkinThickness'])

print("Polimorfismo -- aplicando transformadores en orden:")
for t in transformadores:
    df_poly = t.aplicar(df_poly)
    print(f"  {t.__class__.__name__:<20} aplicado")

print(f"\nDataset tras transformadores: {df_poly.shape}")
print()



# Pipeline: composicion + polimorfismo
# recibe etapas y las ejecuta en orden sin saber cual es cual
class Pipeline:
    def __init__(self, etapas: list):
        self.etapas = etapas  # composicion: el pipeline tiene etapas

    def ejecutar(self, df):
        for etapa in self.etapas:   # polimorfismo: misma llamada para cada etapa
            df = etapa.aplicar(df)
        return df


# Etapa del nucleo algoritmico F3: ordena una columna con merge_sort
class OrdenarColumna(Transformador):
    def __init__(self, columna: str, col_destino: str):
        self.columna     = columna
        self.col_destino = col_destino

    def aplicar(self, df):
        valores = df[self.columna].dropna().tolist()
        ordenados = merge_sort(valores)
        n_nulos = len(df) - len(ordenados)
        df[self.col_destino] = ordenados + [None] * n_nulos
        return df


# Pipeline que une preprocesamiento F2 con nucleo algoritmico F3
df_pipe = df_raw.copy()
df_pipe = castear_bloodpressure(df_pipe)
df_pipe = reemplazar_invalidos(df_pipe)
df_pipe = eliminar_duplicados(df_pipe)
df_pipe = df_pipe.drop(columns=["SkinThickness"])

pipe = Pipeline([
    ImputarMediana(COLS_IMPUTAR),
    EscalarZScore(COLS_ESCALAR),
    OrdenarColumna("Age", "Age_ordenada"),
])

df_pipe = pipe.ejecutar(df_pipe)
print("Pipeline F2 + F3 ejecutado")
print(f"  Etapas ejecutadas : {len(pipe.etapas)}")
print(f"  Age_ordenada (5 primeros): {df_pipe['Age_ordenada'].dropna().head().tolist()}")
print()

# Pipeline completo con encapsulamiento y method chaining
RUTA_DESTINO = Path('../data/processed/diabetes_procesado.csv')

pipeline = PreprocesadorDiabetes(RUTA_RAW)

df_final = (pipeline
            .cargar_datos()
            .eliminar_columnas(['SkinThickness'])
            .castear_bloodpressure()
            .reemplazar_invalidos()
            .eliminar_duplicados()
            .imputar_mediana(COLS_IMPUTAR)
            .crear_variables_derivadas()
            .codificar_categoricas(['AgeGroup', 'BMI_Category'])
            .escalar_variables(COLS_ESCALAR)
            .validar_dataset_final()
            .resumen_comparativo()
            .exportar_dataset(RUTA_DESTINO)
            .obtener_dataframe())

print(f"Dataset final: {df_final.shape[0]} filas x {df_final.shape[1]} columnas")
df_final.head(3)

[CASTING BloodPressure]
  Tipo resultante : float64
  NaN en BP       : 20  (de '?' y valores no numéricos)
[CEROS IMPOSIBLES → NaN]
  Glucose             :   13 ceros reemplazados
  BloodPressure       :   70 ceros reemplazados
  BMI                 :   17 ceros reemplazados
  Insulin             :  634 ceros reemplazados

[VALORES CENTINELA / OUTLIERS EXTREMOS → NaN]
  Glucose > 300             :   12 → NaN
  Age > 100 (centinela 150) :   15 → NaN
  BMI > 60                  :    2 → NaN
  BMI < 10 (error digitación):    4 → NaN

  NaN totales tras este paso: 1196
[DUPLICADOS]  Antes: 2045 | Eliminadas: 45 | Después: 2000
Polimorfismo -- aplicando transformadores en orden:
  ImputarMediana       aplicado
  EscalarZScore        aplicado

Dataset tras transformadores: (2000, 8)

[CASTING BloodPressure]
  Tipo resultante : float64
  NaN en BP       : 20  (de '?' y valores no numéricos)
[CEROS IMPOSIBLES → NaN]
  Glucose             :   13 ceros reemplazados
  BloodPressure       :   70 

,Pregnancies,Glucose,BloodPressure,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,AgeGroup,BMI_Category,AgeGroup_30-44,AgeGroup_45-59,AgeGroup_60+,BMI_Category_Normal,BMI_Category_Sobrepeso,BMI_Category_Obesidad
0,0.622290,0.836101,-0.018168,-0.152113,0.190760,0.454375,1.468962,1,45-59,Obesidad,0,1,0,0,0,1
1,-0.862181,-1.211334,-0.481046,-0.152113,-0.835465,-0.363126,-0.172143,0,30-44,Sobrepeso,1,0,0,0,1,0
2,1.216079,1.973564,-0.635339,-0.152113,-1.319257,0.587663,-0.085769,1,30-44,Normal,1,0,0,1,0,0


## 8. Documentación de arquitectura del código (IV) <a id="8"></a>

### Tabla de componentes y responsabilidades

| Componente | Tipo | Responsabilidad | Ubicación |
|---|---|---|---|
| `cargar_datos(ruta)` | Función F2 | Leer CSV con sep=`;` y validar que no esté vacío | `F3/src/utils.py` |
| `castear_bloodpressure(df)` | Función F2 | Convertir BloodPressure de str a float64; tratar unidades embebidas y `?` | `F3/src/utils.py` |
| `reemplazar_invalidos(df)` | Función F2 | Reemplazar ceros biológicamente imposibles y centinelas por NaN | `F3/src/utils.py` |
| `eliminar_duplicados(df)` | Función F2 | Eliminar filas idénticas y resetear el índice | `F3/src/utils.py` |
| `imputar_mediana(df, cols)` | Función F2 | Rellenar NaN con la mediana de cada columna indicada | `F3/src/utils.py` |
| `crear_variables_derivadas(df)` | Función F2 | Generar AgeGroup y BMI_Category según rangos clínicos estándar | `F3/src/utils.py` |
| `codificar_categoricas(df, cols)` | Función F2 | OHE con drop_first=True para evitar multicolinealidad | `F3/src/utils.py` |
| `escalar_variables(df, cols)` | Función F2 | Estandarización Z-score con StandardScaler | `F3/src/utils.py` |
| `validar_dataset_final()` | Método POO (F2 como base) | Asserts de integridad: NaN, duplicados, Outcome, media y std de escaladas. Se llama sin parámetros sobre la instancia de la clase. | `F3/src/utils.py` |
| `merge_sort(lista)` | Función F3 | Ordenamiento recursivo O(n log n) -- divide y vencerás | Notebook 3.1 |
| `_combinar(izq, der)` | Auxiliar F3 | Fusión de dos listas ordenadas en O(n) | Notebook 3.1 |
| `busqueda_binaria_recursiva(...)` | Función F3 | Búsqueda recursiva O(log n) sobre lista ordenada | Notebook 3.2 |
| `busqueda_lineal(lista, obj)` | Función F3 | Búsqueda iterativa O(n) para comparación en medición 4 | Notebook 3.3 |
| `auditar_carpetas_recursivo(ruta)` | Función F3 | Recorrido recursivo del árbol de directorios del proyecto | `F3/src/utils.py` |
| `PreprocesadorDiabetes` | Clase POO | Pipeline completo encapsulado con 14 métodos y method chaining | `F3/src/utils.py` |


### Justificación de las decisiones de diseño

| Decisión | Por qué | Alternativa descartada |
|---|---|---|
| **Mediana para imputar** | Glucose, BMI e Insulin tienen outliers extremos (Glucose=999, Age=150). La mediana no se ve afectada por esos valores; la media los incorpora y sesga el resultado. | Media aritmética |
| **StandardScaler** | Los modelos que usaremos en F4 (SVM, regresión logística, PCA) requieren variables con media=0 y std=1. MinMaxScaler comprime todo a [0,1] pero un outlier residual aplasta el rango útil. | MinMaxScaler |
| **Merge Sort** | O(n log n) en todos los casos. Insertion Sort y Bubble Sort son O(n^2) en el peor caso: para n=2.000, 4.000.000 vs 22.000 operaciones. | Insertion Sort, Bubble Sort |
| **Búsqueda binaria** | Demuestra el paradigma recursivo con O(log n) verificable. Un dict daría O(1) pero no es recursivo, que es el requisito de la fase. | dict, set |
| **drop_first=True en OHE** | Evita la multicolinealidad perfecta: k categorías con drop_first producen k-1 columnas, suficientes para codificar toda la información. | OHE sin drop_first |
| **POO con self.df** | Elimina la dependencia de variables globales en el notebook. Con funciones sueltas, `df` puede sobrescribirse al ejecutar celdas fuera de secuencia. | Solo funciones independientes |
| **SkinThickness eliminada en F3** | El 32.6% de sus registros son ceros imposibles. Imputar sobre datos de tan baja calidad introduciría sesgo. Se mantiene el criterio de F2: eliminación explícita con `.eliminar_columnas()`. | Recuperar con imputación |

### Vinculación con el repositorio

| Artefacto | Ubicación |
|---|---|
| Este notebook | `F3/notebooks/F3_Preprocesamiento.ipynb` |
| Scripts (funciones + clases) | `F3/src/utils.py` |
| Dataset fuente | `F3/data/raw/diabetes_raw.csv` |
| Dataset procesado | `F3/data/processed/diabetes_procesado.csv` |
| Fase 2 (referencial) | `F2/notebooks/` + `F2/src/utils.py` |
| README | `README.md` en la raiz del repositorio -- instrucciones de ejecucion |
| Repositorio | https://github.com/MagUnab/mcdia500-programacion-cd-g6 -- rama `feature/fase3-poo` |

## 9. Bibliografía (VII) <a id="9"></a>

Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). *Introduction to algorithms* (4.a ed.). MIT Press.  
Fundamento del analisis de complejidad Big-O, Merge Sort y busqueda binaria usados en las Secciones 3 y 5.

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.a ed.). O'Reilly Media.  
Base tecnica para las operaciones vectorizadas de Pandas frente a bucles Python (Seccion 5.1) y para el pipeline de preprocesamiento (Secciones 2 y 7).

Python Software Foundation. (2024). *The Python standard library*. https://docs.python.org/3/library/  
Modulos `timeit` (Mediciones 1, 2 y 4) y `tracemalloc` (Medicion 3) para la medicion reproducible de complejidad temporal y espacial.

pandas Development Team. (2024). *pandas documentation: User guide*. https://pandas.pydata.org/docs/  
Documentacion de `pd.read_csv()`, `str.replace()` vectorizado y `DataFrame.dropna()` utilizados en las Secciones 2 y 5.

Scikit-learn developers. (2024). *Scikit-learn: Machine learning in Python -- preprocessing.StandardScaler*. https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html  
Referencia del escalamiento Z-score aplicado en la Seccion 7 y justificado en la Seccion 8.

MCDI500. (2025). *Guia de desarrollo -- Sumativa 3, Fase 3*. Universidad Andres Bello, Magister en Ciencia de Datos e Inteligencia Artificial.  
Guia de orientacion para la implementacion del nucleo algoritmico, la integracion de POO y las mediciones de complejidad de esta fase.

MCDI500. (2025). *Guia de desarrollo -- Formativa 3, Fase 3*. Universidad Andres Bello, Magister en Ciencia de Datos e Inteligencia Artificial.  
Referencia para la estructura del notebook, la rubrica de evaluacion y los criterios de codificacion funcional aplicados en este entregable.
